# Interactive NIPALS for PCA (2D)

This notebook demonstrates **NIPALS** for extracting the **first principal component** (PC1) of a centered data matrix.

**What you can do here**:
- Step through NIPALS iterations with a **Plotly slider**
- Compare the iterated direction **pₖ** to the **true PC1** from an eigendecomposition
- Plot convergence (angle to true PC1)
- Experiment with different initializations and dataset shapes

> Tip: Run cells top-to-bottom. Plotly figures will appear inline in Jupyter/VS Code.


In [ ]:
import numpy as np
import plotly.graph_objects as go

np.set_printoptions(precision=4, suppress=True)


## 1) Generate a tilted 2D dataset

We use a 2D Gaussian with correlated variables so the first principal component is visually clear.


In [ ]:
def generate_data(n=300, cov_scale=1.0, seed=42):
    rng = np.random.default_rng(seed)
    mean = np.array([0.0, 0.0])
    cov = cov_scale * np.array([[3.0, 2.0],
                                [2.0, 2.0]])
    X = rng.multivariate_normal(mean, cov, size=n)
    X = X - X.mean(axis=0)  # center
    return X

X = generate_data(n=300, cov_scale=1.0, seed=42)
X.shape


Quick scatter plot of the centered data.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=X[:,0], y=X[:,1], mode='markers', marker=dict(size=5), name='Data'))
fig.update_layout(title='Centered data (2D)', xaxis_title='x1', yaxis_title='x2', width=700, height=600)
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.show()


## 2) True PC1 (for comparison)

For centered data, PC1 is the eigenvector corresponding to the largest eigenvalue of **XᵀX** (proportional to covariance).

In [ ]:
def true_pc1_direction(X):
    C = X.T @ X
    eigvals, eigvecs = np.linalg.eigh(C)  # symmetric
    idx = np.argmax(eigvals)
    v = eigvecs[:, idx]
    return v / np.linalg.norm(v), eigvals[idx]

true_pc1, lambda1 = true_pc1_direction(X)
true_pc1, lambda1


## 3) NIPALS (PC1) with iteration history

NIPALS update equations (one common PCA form):

- Initialize score vector **t** (e.g., a column of X)
- Iterate until convergence:
  - **p = Xᵀ t / (tᵀ t)**, then normalize p
  - **t = X p / (pᵀ p)**

We store each iterate **pₖ** so we can animate it.

In [ ]:
def angle_degrees(u, v):
    # Angle between directions (treat ±v as the same direction for PCA)
    u = u / np.linalg.norm(u)
    v = v / np.linalg.norm(v)
    c = float(np.dot(u, v))
    c = np.clip(abs(c), -1.0, 1.0)
    return float(np.degrees(np.arccos(c)))

def nipals_pc1_history(X, n_iter=30, init_t=None):
    # init_t should be length n (scores). If None, use first column of X.
    if init_t is None:
        t = X[:, 0].copy()
    else:
        t = np.array(init_t, dtype=float).copy()
    
    hist = []
    for k in range(n_iter):
        denom_t = float(t.T @ t)
        if denom_t == 0:
            raise ValueError('Initialization produced t with zero norm.')
        p = (X.T @ t) / denom_t
        p_norm = np.linalg.norm(p)
        if p_norm == 0:
            raise ValueError('Computed p has zero norm; try different init or data.')
        p = p / p_norm

        denom_p = float(p.T @ p)  # should be 1 after normalization
        t = (X @ p) / denom_p

        hist.append({'k': k, 't': t.copy(), 'p': p.copy()})
    return hist

history = nipals_pc1_history(X, n_iter=30)
history[0]['p'], history[-1]['p']


## 4) Interactive animation (slider)

The slider lets students step through iterations and watch the direction **pₖ** rotate toward the true PC1.

In [ ]:
def nipals_animation_2d(X, history, true_pc1, vector_scale=5.0, title='NIPALS convergence to PC1'):
    frames = []
    
    for step in history:
        k = step['k']
        p = step['p']
        ang = angle_degrees(p, true_pc1)
        
        frames.append(
            go.Frame(
                name=str(k),
                data=[
                    go.Scatter(
                        x=X[:, 0], y=X[:, 1],
                        mode='markers',
                        marker=dict(size=5),
                        name='Data'
                    ),
                    go.Scatter(
                        x=[0, vector_scale * p[0]],
                        y=[0, vector_scale * p[1]],
                        mode='lines',
                        line=dict(width=5),
                        name='Current p'
                    ),
                    go.Scatter(
                        x=[0, vector_scale * true_pc1[0]],
                        y=[0, vector_scale * true_pc1[1]],
                        mode='lines',
                        line=dict(dash='dash', width=5),
                        name='True PC1'
                    ),
                ],
                layout=go.Layout(
                    title=f"{title} — iteration {k}, angle to true PC1: {ang:.3f}°"
                )
            )
        )
    
    fig = go.Figure(data=frames[0].data, frames=frames)
    
    fig.update_layout(
        width=800,
        height=700,
        xaxis_title='x1',
        yaxis_title='x2',
        xaxis=dict(zeroline=False),
        yaxis=dict(zeroline=False),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
        sliders=[{
            'active': 0,
            'currentvalue': {'prefix': 'Iteration: '},
            'pad': {'t': 50},
            'steps': [
                {'method': 'animate',
                 'args': [[f.name], {'frame': {'duration': 0, 'redraw': True}, 'mode': 'immediate'}],
                 'label': f.name}
                for f in frames
            ]
        }],
        updatemenus=[{
            'type': 'buttons',
            'direction': 'left',
            'x': 0.0,
            'y': 1.15,
            'buttons': [
                {'label': 'Play',
                 'method': 'animate',
                 'args': [None, {'frame': {'duration': 300, 'redraw': True}, 'fromcurrent': True}]},
                {'label': 'Pause',
                 'method': 'animate',
                 'args': [[None], {'frame': {'duration': 0, 'redraw': False}, 'mode': 'immediate'}]}
            ]
        }]
    )
    
    fig.update_yaxes(scaleanchor='x', scaleratio=1)
    fig.show()

nipals_animation_2d(X, history, true_pc1)


## 5) Convergence diagnostic: angle to true PC1

This plot shows how quickly the NIPALS direction aligns with the true eigenvector.

In [ ]:
angles = [angle_degrees(step['p'], true_pc1) for step in history]

fig = go.Figure()
fig.add_trace(go.Scatter(y=angles, mode='lines+markers', name='Angle'))
fig.update_layout(
    title='Angle between NIPALS p_k and true PC1',
    xaxis_title='Iteration k',
    yaxis_title='Angle (degrees)',
    width=800,
    height=450
)
fig.show()


## 6) Experiment: different initializations

NIPALS is closely related to the **power method**, so it typically converges from most non-degenerate initial vectors.


In [ ]:
rng = np.random.default_rng(0)
random_init_t = rng.standard_normal(X.shape[0])
history_rand = nipals_pc1_history(X, n_iter=30, init_t=random_init_t)

nipals_animation_2d(X, history_rand, true_pc1, title='NIPALS (random init) convergence to PC1')


## 7) Experiment: change the data shape (covariance strength)

When the top two eigenvalues are close, convergence is slower.


In [ ]:
X2 = generate_data(n=300, cov_scale=0.35, seed=7)
true_pc1_X2, _ = true_pc1_direction(X2)
history_X2 = nipals_pc1_history(X2, n_iter=40)

nipals_animation_2d(X2, history_X2, true_pc1_X2, title='NIPALS convergence (weaker covariance)')


## 8) (Optional) Extract PC2 via deflation

After finding PC1 (scores **t** and loading **p**), a common deflation step is:

\[
X \leftarrow X - t p^T
\]

Then run NIPALS again on the deflated matrix to get PC2.


In [ ]:
def deflate(X, t, p):
    return X - np.outer(t, p)

# Use the final iterate from the first run
t1 = history[-1]['t']
p1 = history[-1]['p']
X_def = deflate(X, t1, p1)

true_pc1_def, _ = true_pc1_direction(X_def)
history_pc2 = nipals_pc1_history(X_def, n_iter=30)

nipals_animation_2d(X_def, history_pc2, true_pc1_def, title='NIPALS on deflated X (next component)')


## Notes for teaching

- NIPALS updates are closely related to **power iteration** on \(X^T X\).
- The slider animation makes the **iterative refinement** tangible.
- Deflation illustrates why PCA components are extracted one-by-one.
